# Configuration space and degrees of freedom

The numbers of the Note: the floor robot's configuration and the points of its body, the circle it drives and where its heading wraps, degree-of-freedom counts, Grübler's formula, the arm's hand, explicit and implicit headings, the pole, and the planned path to the door. The model lives in `images/cmodel.py`, which the figures use too.

In [1]:
import sys
import numpy as np
sys.path.insert(0, 'images')
from cmodel import (Q0, R_BODY, TWO_PI, body_point, circle_drive, wrap, arm, grubler, plan_configs, clearance,
                    WAY, TH_GOAL)
print('configuration q =', np.round(Q0, 3))

configuration q = [2.    1.    0.524]


## Every point of the body follows from (x, y, theta)

In [2]:
front = body_point(Q0, (R_BODY, 0))
left = body_point(Q0, (0, R_BODY))
print('front point', np.round(front, 4), ' left point', np.round(left, 3))
print('distance front-left', round(float(np.hypot(front[0] - left[0], front[1] - left[1])), 3))

front point [2.2165 1.125 ]  left point [1.875 1.217]
distance front-left 0.354


## The circle of RO-001 in configuration space

In [3]:
t_wrap = (TWO_PI - Q0[2]) / 1.0
print('heading reaches 2 pi after', round(t_wrap, 4), 's')
print('one full turn takes', round(TWO_PI, 3), 's; end configuration', np.round(circle_drive(TWO_PI)[:2], 6),
      'heading wrapped', round(float(wrap(circle_drive(TWO_PI)[2])), 3))

heading reaches 2 pi after 5.7596 s
one full turn takes 6.283 s; end configuration [2. 1.] heading wrapped 0.524


## Degrees of freedom: points minus independent constraints

In [4]:
print('plane: 3 points x 2 numbers - 3 distances =', 3 * 2 - 3)
print('space: 3 points x 3 numbers - 3 distances =', 3 * 3 - 3)
print('a fourth point D in the plane: 2 numbers - 2 independent distances =', 2 - 2)

plane: 3 points x 2 numbers - 3 distances = 3
space: 3 points x 3 numbers - 3 distances = 6
a fourth point D in the plane: 2 numbers - 2 independent distances = 0


## Grübler's formula

In [5]:
print('2R arm      ', grubler(3, 3, 2, [1, 1]))
print('floor robot ', grubler(3, 2, 0, []))
print('closed loop ', grubler(3, 4, 4, [1, 1, 1, 1]))
print('3R arm      ', grubler(3, 4, 3, [1, 1, 1]))
print('robot + arm  3 + 2 =', grubler(3, 2, 0, []) + grubler(3, 3, 2, [1, 1]))

2R arm       2
floor robot  3
closed loop  1
3R arm       3
robot + arm  3 + 2 = 5


## The arm's hand at (30, 60) degrees

In [6]:
e, h = arm(np.radians(30), np.radians(60))
print('elbow', np.round(e, 3), ' hand', np.round(h, 3))

elbow [0.26 0.15]  hand [0.26 0.35]


## Explicit and implicit headings across 360 degrees

In [7]:
for d in (359, 1):
    th = np.radians(d)
    print(f'{d:3d} deg: theta = {th:.5f} rad, (cos, sin) = ({np.cos(th):.4f}, {np.sin(th):.4f})')
a, b = np.radians(359), np.radians(1)
print('theta changes by', round(b - a, 5), 'rad;  the point (cos, sin) moves',
      round(float(np.hypot(np.cos(b) - np.cos(a), np.sin(b) - np.sin(a))), 4))

359 deg: theta = 6.26573 rad, (cos, sin) = (0.9998, -0.0175)
  1 deg: theta = 0.01745 rad, (cos, sin) = (0.9998, 0.0175)
theta changes by -6.24828 rad;  the point (cos, sin) moves 0.0349


In [8]:
a, b = np.radians(350), np.radians(10)
print('350 and 10 deg: theta', round(a, 5), round(b, 5), 'difference', round(a - b, 5))
print('implicit points', np.round([np.cos(a), np.sin(a)], 5), np.round([np.cos(b), np.sin(b)], 5),
      'apart', round(float(np.hypot(np.cos(a) - np.cos(b), np.sin(a) - np.sin(b))), 4), '; 20 deg =', round(np.radians(20), 3), 'rad')

350 and 10 deg: theta 6.10865 0.17453 difference 5.93412
implicit points [ 0.98481 -0.17365] [0.98481 0.17365] apart 0.3473 ; 20 deg = 0.349 rad


## The pole: longitude change for 1 m walked east

In [9]:
for rho in (10, 1):
    print(f'{rho:2d} m from the pole: {1 / rho:.1f} rad = {np.degrees(1 / rho):.1f} degrees')

10 m from the pole: 0.1 rad = 5.7 degrees
 1 m from the pole: 1.0 rad = 57.3 degrees


## The planned motion from the start to the door

In [10]:
cfg, legs = plan_configs()
print('leg headings (deg)', np.round(np.degrees(legs), 2))
print('leg lengths (m)', [round(float(np.hypot(b[0] - a[0], b[1] - a[1])), 3) for a, b in zip(WAY, WAY[1:])])
print('goal configuration', WAY[-1], round(TH_GOAL, 3))
c = clearance()
print('smallest distance from the centre path to an obstacle or wall', round(c, 3), 'm; body clearance', round(c - R_BODY, 3), 'm')

leg headings (deg) [61.93]
leg lengths (m) [3.4]
goal configuration (3.6, 4.0) 1.081
smallest distance from the centre path to an obstacle or wall 0.382 m; body clearance 0.132 m
